**Date**: June 28, 2025

**Author**: Zoey Liu

**Purpose** A script for automatic checking of some, but not all, sentence-level annotation errors for Storiza project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls

In [ ]:
%cd drive/MyDrive/Colab\ Notebooks/

In [2]:
import pandas as pd
import json, ast
import statistics
#from difflib import SequenceMatcher

**Sentence-level checks**

In [ ]:
## Reading data
original_data = pd.read_csv('export_157513_project-157513-at-2025-07-01-07-44-30c2d71c.csv', delimiter = '\t')
original_data.columns
original_data.head(n=8)
original_id_list = original_data['id'].tolist()

## Filter out cases with no annotations for sentence segmentation (problematic cases)
data = original_data[~(original_data['SentenceLabel'].isna())]

In [ ]:
## Cross-annotated tasks will appear the number of times that correspond to how many times it's been annotated
print(len(original_data), len(data))
print('\n')
for id in original_id_list:
  if id not in id_list:
    print(id)

In [ ]:
## Loading some variables
SentenceLabel_list = data['SentenceLabel'].tolist() ## Timestamps/Segmentations
SentenceSelect_list = data['SentenceSelect'].tolist() ## Annotated intended sentences
goldStandardText_list = data['goldStandardText'].tolist() ## Original gold standard texts from story.xlsx
annotator_list = data['annotator'].tolist() ## Annotators
id_list = data['id'].tolist() ## Audio ID showing up on the UI
possibleSentences_list = data['possibleSentences'].tolist() ## Use this as the most updated reference sentence segmentation used as the Intended Sentence in the UI (the list where the annotators chose from)
issues_list = data['issues'].tolist() # Comments from annotators
matching_file_list = data['matching_file'].tolist()

In [ ]:
## Generate a list of audio files with no annotations
original_id_list = original_data['id'].tolist()
unannotated_audios = list(set(original_id_list) - set(id_list))

In [ ]:
## Do a sample check
print("Example check")
print("Timstamps: ", SentenceLabel_list[0])
print("Intended sentences: ", SentenceSelect_list[0])
print("Goldstandard sentences: ", goldStandardText_list[0])

In [ ]:
for i in range(len(data)):

  ## Annotated time stamps
  SentenceLabel_str = ''
  try:
    SentenceLabel_str = json.loads(SentenceLabel_list[i])
  except:
    SentenceLabel_str = ast.literal_eval(SentenceLabel_list[i])

  ## Sort timestamps by start time
#  timestamps_sorted = sorted(SentenceLabel_str, key=lambda x: float(x['start']))

  ## Annotated Intended Sentences
  selected_sentences = ''
  try:
    selected_sentences = json.loads(SentenceSelect_list[i])
  except:
    try:
      selected_sentences = ast.literal_eval(SentenceSelect_list[i])
    except:
      ## When an audio has just one utterance, its intended sentence is returned as a single string, not a list with one string
      selected_sentences = [SentenceSelect_list[i]]

    # Sorting timestamps and annotated intended sentences together (ignore "Other")
  combined = [
    (ts, s) for ts, s in zip(SentenceLabel_str, selected_sentences) # This works even when lengths of timestamps and annotated intended sentences are different; therefore this step is after the Check above
    if s != "Other"
  ]

  # Sort by start time
  combined_sorted = sorted(combined, key=lambda x: float(x[0]["start"]))
  timestamps_sorted = [ts for ts, _ in combined_sorted]
  selected_sentences_sorted = [s for _, s in combined_sorted]

  ## Original old standard sentences
  goldStandardText_str = goldStandardText_list[i]
  gold_sentences = [line.split('. ', 1)[1] for line in goldStandardText_str.strip().split('\n')]

  ## Actual list of sentence segmentation used in the UI
  possibleSentence_str = json.loads(possibleSentences_list[i])
  possible_sentences = [item["value"] for item in possibleSentence_str][ : -1] # Remove 'Other'

  ## Checking if any segmentation does not have an Intended Sentence; this includes cases such as 'Other'
  ## If so, the number of timestamps would not match the number of intended sentences
  if len(SentenceLabel_str) != len(selected_sentences):
    print(id_list[i], len(SentenceLabel_str), len(selected_sentences), "Some segmentation does not have an Intended Sentence")
    print(possible_sentences)
    print(selected_sentences_sorted)
    print('\n')

In [ ]:
for i in range(len(data)):

  ## Annotated time stamps
  SentenceLabel_str = ''
  try:
    SentenceLabel_str = json.loads(SentenceLabel_list[i])
  except:
    SentenceLabel_str = ast.literal_eval(SentenceLabel_list[i])

  ## Sort timestamps by start time
#  timestamps_sorted = sorted(SentenceLabel_str, key=lambda x: float(x['start']))

  ## Annotated Intended Sentences
  selected_sentences = ''
  try:
    selected_sentences = json.loads(SentenceSelect_list[i])
  except:
    try:
      selected_sentences = ast.literal_eval(SentenceSelect_list[i])
    except:
      ## When an audio has just one utterance, its intended sentence is returned as a single string, not a list with one string
      selected_sentences = [SentenceSelect_list[i]]

  ## Checking if there are any duplicates within a list of intended sentences
  ## There are two possible case scenarios:
  ## (1) A sentence was repeated in child production
  ## (2) Identify cases with the first type of sentence segmentation error where the sentence should be split into two
  ## RESULTS thus far: Additionally idiosyncratic case: 190669709 Sentence segmentation error with the first listed intended sentence. It contains the title of the story, which is not read. Further, the title is accidentally combined with the first sentence of the story, which the child also does not read. "Title: Minnie's Spring Morning Minnie woke up early one sunny spring morning."
  temp_selected_sentences = [item for item in selected_sentences if item != 'Other']
  try:
    assert len(temp_selected_sentences) == len(set(temp_selected_sentences))
  except:
    print(id_list[i], len(temp_selected_sentences), len(set(temp_selected_sentences)))
    for sent in list(set(temp_selected_sentences)):
      if temp_selected_sentences.count(sent) != 1:
        print(sent)
        print('\n')
    print('\n')

In [ ]:
for i in range(len(data)):

  ## Annotated time stamps
  SentenceLabel_str = ''
  try:
    SentenceLabel_str = json.loads(SentenceLabel_list[i])
  except:
    SentenceLabel_str = ast.literal_eval(SentenceLabel_list[i])

  ## Sort timestamps by start time
#  timestamps_sorted = sorted(SentenceLabel_str, key=lambda x: float(x['start']))

  ## Annotated Intended Sentences
  selected_sentences = ''
  try:
    selected_sentences = json.loads(SentenceSelect_list[i])
  except:
    try:
      selected_sentences = ast.literal_eval(SentenceSelect_list[i])
    except:
      ## When an audio has just one utterance, its intended sentence is returned as a single string, not a list with one string
      selected_sentences = [SentenceSelect_list[i]]

  # Sorting timestamps and annotated intended sentences together (ignore "Other")
  combined = [
    (ts, s) for ts, s in zip(SentenceLabel_str, selected_sentences) # This works even when lengths of timestamps and annotated intended sentences are different; therefore this step is after the Check above
    if s != "Other"
  ]

  # Sort by start time
  combined_sorted = sorted(combined, key=lambda x: float(x[0]["start"]))
  timestamps_sorted = [ts for ts, _ in combined_sorted]
  selected_sentences_sorted = [s for _, s in combined_sorted]

  ## Original old standard sentences
  goldStandardText_str = goldStandardText_list[i]
  gold_sentences = [line.split('. ', 1)[1] for line in goldStandardText_str.strip().split('\n')]

  ## Actual list of sentence segmentation used in the UI
  possibleSentence_str = json.loads(possibleSentences_list[i])
  possible_sentences = [item["value"] for item in possibleSentence_str][ : -1] # Remove 'Other'

  ## Checking if child reverses any sentence order in production, regardless of whether they produced all sentences from the Intended Sentence list
  index_list = []
  temp_selected_sentences_sorted = []
  for z in range(len(selected_sentences_sorted)):
    sent = selected_sentences_sorted[z]
    if sent not in temp_selected_sentences_sorted:
      temp_selected_sentences_sorted.append(sent)

  for z in range(len(selected_sentences_sorted)):
    index_list.append(possible_sentences.index(selected_sentences_sorted[z]))

  new_index_list = sorted(index_list)
  if new_index_list != index_list:
    print(id_list[i], "Child reverses sentence order", matching_file_list[i])
    print(index_list)
    print(new_index_list)
    print('\n')

In [ ]:
## cases identified by Michael but was not in my list; they actually do not have reverse ordering
specific_cases = ['uid_HGLxgWVfJZWhAsqMf761d40h3eQ2_sid_DmEr4pqsDFri8xozTdsS_1743276724.wav',
'uid_gUfM3yODstSdwltidtEzSlIo0q93_sid_yYkurtcTpL0BYwYKWgIi_1742427844.wav',
'uid_ovpeape9PZbAq7VLrirwGOYrvsh1_sid_vrmxG2Pc3NPfDSVnUg5r_1742390017.wav',
'uid_HGLxgWVfJZWhAsqMf761d40h3eQ2_sid_F4tWmApCfflaPRfdT4xX_1741271551.wav',
'uid_Gt0cspsUtRSvEXXHZKUWlVAfyVq2_sid_qKQmMKuXG2FOUvLlR8Nk_1742062228.wav'
]

for tok in matching_file_list:
  if tok in specific_cases:
    index = matching_file_list.index(tok)
    print(id_list[index])

In [ ]:
for i in range(len(data)):

  ## Annotated time stamps
  SentenceLabel_str = ''
  try:
    SentenceLabel_str = json.loads(SentenceLabel_list[i])
  except:
    SentenceLabel_str = ast.literal_eval(SentenceLabel_list[i])

  ## Sort timestamps by start time
#  timestamps_sorted = sorted(SentenceLabel_str, key=lambda x: float(x['start']))

  ## Annotated Intended Sentences
  selected_sentences = ''
  try:
    selected_sentences = json.loads(SentenceSelect_list[i])
  except:
    try:
      selected_sentences = ast.literal_eval(SentenceSelect_list[i])
    except:
      ## When an audio has just one utterance, its intended sentence is returned as a single string, not a list with one string
      selected_sentences = [SentenceSelect_list[i]]

  # Sorting timestamps and annotated intended sentences together (ignore "Other")
  combined = [
    (ts, s) for ts, s in zip(SentenceLabel_str, selected_sentences) # This works even when lengths of timestamps and annotated intended sentences are different; therefore this step is after the Check above
    if s != "Other"
  ]

  # Sort by start time
  combined_sorted = sorted(combined, key=lambda x: float(x[0]["start"]))
  timestamps_sorted = [ts for ts, _ in combined_sorted]
  selected_sentences_sorted = [s for _, s in combined_sorted]

  ## Original old standard sentences
  goldStandardText_str = goldStandardText_list[i]
  gold_sentences = [line.split('. ', 1)[1] for line in goldStandardText_str.strip().split('\n')]

  ## Actual list of sentence segmentation used in the UI
  possibleSentence_str = json.loads(possibleSentences_list[i])
  possible_sentences = [item["value"] for item in possibleSentence_str][ : -1] # Remove 'Other'

  ## Checking if any sentence from the Intended Sentence list was not chosen
  ## There are two possible case scenarios:
  ## (1) The child skips a sentence
  ## (2) Identify SOME OF the cases with the second sentence segmentation error where a sentence is erroneously split into two in the Intended Sentence List
  if len(possible_sentences) > len(selected_sentences_sorted):
    case_issue = issues_list[i]
    has_segmentation_error = 'no'
    if type(case_issue) is not float:
      if 'segment' in case_issue or 'Segment' in case_issue or 'split' in case_issue or 'Split' in case_issue:
        has_segmentation_error = 'yes'

    if has_segmentation_error == 'yes':
      print(id_list[i])
      for z in range(len(possible_sentences)):
        try:
          if possible_sentences[z] == selected_sentences_sorted[z]:
            print('SAME', possible_sentences[z], selected_sentences_sorted[z])
          else:
            print('Different', possible_sentences[z], selected_sentences_sorted[z])
        except:
          print(possible_sentences[z])


      print('\n')

In [ ]:
for i in range(len(data)):

  ## Annotated time stamps
  SentenceLabel_str = ''
  try:
    SentenceLabel_str = json.loads(SentenceLabel_list[i])
  except:
    SentenceLabel_str = ast.literal_eval(SentenceLabel_list[i])

  ## Sort timestamps by start time
#  timestamps_sorted = sorted(SentenceLabel_str, key=lambda x: float(x['start']))

  ## Annotated Intended Sentences
  selected_sentences = ''
  try:
    selected_sentences = json.loads(SentenceSelect_list[i])
  except:
    try:
      selected_sentences = ast.literal_eval(SentenceSelect_list[i])
    except:
      ## When an audio has just one utterance, its intended sentence is returned as a single string, not a list with one string
      selected_sentences = [SentenceSelect_list[i]]


  # Sorting timestamps and annotated intended sentences together (ignore "Other")
  combined = [
    (ts, s) for ts, s in zip(SentenceLabel_str, selected_sentences) # This works even when lengths of timestamps and annotated intended sentences are different; therefore this step is after the Check above
    if s != "Other"
  ]

  # Sort by start time
  combined_sorted = sorted(combined, key=lambda x: float(x[0]["start"]))
  timestamps_sorted = [ts for ts, _ in combined_sorted]
  selected_sentences_sorted = [s for _, s in combined_sorted]

  ## Original old standard sentences
  goldStandardText_str = goldStandardText_list[i]
  gold_sentences = [line.split('. ', 1)[1] for line in goldStandardText_str.strip().split('\n')]

  ## Actual list of sentence segmentation used in the UI
  possibleSentence_str = json.loads(possibleSentences_list[i])
  possible_sentences = [item["value"] for item in possibleSentence_str][ : -1] # Remove 'Other'

  ## Checking if there is segmentation overlap between two adjacent utterances
  if len(SentenceLabel_str) == len(selected_sentences_sorted):
    for z in range(1, len(timestamps_sorted)): # for the current segmentation, compares it to the previous one
      prev_end = float(timestamps_sorted[z - 1]['end'])
      curr_start = float(timestamps_sorted[z]['start'])
      if curr_start < prev_end:
      #    print(f"{id_list[i]} ✗ Overlap between segment {z} and {z+1}: {curr_start:.2f} < {prev_end:.2f}")
        print(f"{id_list[i]} ✗ Overlap between Sentence {z} and {z + 1}") # z refers to the actual order of the sentence in the list of intended sentences, therefore it might not correspond to the segmend ID to the right side of the UI, if the annotators forgot to label a segment then later on added it
        print(annotator_list[i])
        try:
          print(timestamps_sorted[z-1], selected_sentences_sorted[z-1])
          print(timestamps_sorted[z], selected_sentences_sorted[z])
        except:
          print(id_list[i], z)
          print('ERROR')

          print(timestamps_sorted[z], selected_sentences_sorted[z])
          break
        print('\n')


**Additional checks using Word-level csv file**

In [ ]:
word_level_data = pd.read_csv('export_157618_project-157618-at-2025-07-17-08-51-2c7ba368.csv', delimiter = ',')
word_level_data.columns

In [ ]:
id_list = word_level_data['id'].tolist()
sentence_id_list = word_level_data['sentence_level_id'].tolist()
goldStandard_list = word_level_data['goldStandard'].tolist()

In [ ]:
repeated_sentences = [
    "One day, Whiskers found a hat on the mat",
    "The hat was big and flat",
    "Roy was sad but then spotted it on the shore",
    "In a test of speed, he raced against the tallest kids",
    "They cheered as he got faster at each turn",
    "He spies a bright site by the pine",
    "Look, a map",
    "He pulled out a map.",
    "Tim wanted to check",
    "Rusty loved to uncover hidden spots under the big, shady trees",
    "Her art and work are both neat",
    "A gentle breeze brushed his cheek",
    "After leaning back on the grass, he closed his eyes",
    "They loved exploring the endless woods, finding colorful flowers and watching birds",
    "His best friend was a small, mindful rabbit called Rue",
    "The race began, and initially, Pizza zoomed ahead, her toppings dancing with every move",
    "But, as the race progressed"
]

In [ ]:
## Checking if repeated sentences are labeled correctly
repeated_list = word_level_data['repeated'].tolist()
for sent in repeated_sentences:
  n = 0
  for i in range(len(word_level_data)):
    goldStandard = goldStandard_list[i]
#    n = 0
    if sent in goldStandard:
      n += 1
      print(goldStandard, id_list[i], int(sentence_id_list[i]))
      print(repeated_list[i])
  if n == 0:
    print('Did not find this entence')


In [ ]:
## Checking if segments are separated correctly, i.e., (5b)

separate_list = [("As Catsy sails, she meets a shy fish.",
                "Catsy says with a wide smile."), #“Join the ship,” Catsy says with a wide smile.

               ("You are my best friend, Penny, and I will always be here for you.", #Jane said, “You are my best friend, Penny, and I will always be here for you.”
                "And so, Jane and Penny lived happily, sharing many adventures together, day and night."),

               ("Even on rainy days, we can find joy!", #Joy said, “Even on rainy days, we can find joy!”
                "Roy nodded, happy to have such a joyful friend."),

               ("but Tara was there with her charm.",  #It wasn’t hard, but Tara was there with her charm.
                "she said, patting his arm gently with a pad"), #“Don’t worry, Jake, it’s just a little scar,” she said, patting his arm gently with a pad.

              ("She bandaged it fast and gave him a smile that could light up the dark.",
               "Jake beamed like a star as he darted back to class."), #“Thanks, Tara!” Jake beamed like a star as he darted back to class.

              ("Aunt Sue cheered", #Aunt Sue cheered, “Great catch, Patch!”
              "They ended the day with a treat, a yummy batch of cookies."),

              ("It was a sly owl perched on a branch.",
               "he hooted kindly."), #“Can I join?” he hooted kindly.

               ("Dale waved, hoping for a friend.",
                "Want to leap and play?", #“Hi, Dawn!” hooted Dale, “Want to leap and play?”
                "Dawn agreed, her eyes twinkling.")
               ]

In [ ]:
for pair in separate_list:
  for tok in pair:
    for i in range(len(word_level_data)):
      goldStandard = goldStandard_list[i]
      if tok in goldStandard:
        print(goldStandard, int(id_list[i]), int(sentence_id_list[i]))
  print('\n')

In [ ]:
## Checking if segments are expanded correctly, i.e. (6b)

expand_list = ["Roy called out, waving a corner of himself.",
               "Quick, we need to find a place to stay dry!",
               "Phew! We're safe here!",
               "Look at those colors!",
               "This is the best!",
               "Look at it soar!",
               "Look, a map!",
               "What a spot to teach my kittens to jump!",
               "This is fun!",
               "Go, cat!",
               "Would you like to play with us?",
               "search for shells!",
               "At the end, they said,",
               "Caroline laughed",
               "Can I join?",
               "Thanks, Sam!",
               "Hello, snail!",
               "Let's play!",
               "make a kite!",
               "His mother said",
               "Hurrah! My tooth came out!",
               "Come on, Zeus!", # Let’s play!",
               "At home, Zade patted Zeus and said",
               "It's in the box!",
               "What was that?",
               "This cake is so nice!",
               "Let's make it really tall!",
               "Help! My wing is stuck",
               "Where will we go first?"

]

In [ ]:
for tok in expand_list:
  n = 0
  for i in range(len(word_level_data)):
    goldStandard = goldStandard_list[i]
    if tok in goldStandard:
      try:
        print(goldStandard, int(id_list[i]), int(sentence_id_list[i]))
      except:
        print(goldStandard, int(id_list[i]))
      n += 1
  print('\n')

  if n == 0:
    print(tok, 'Did not find this sentence')

In [ ]:
## Checking if segments are combined correctly, i.e., (6c)

combine_list = [
    "Should we push on and find the hidden book?",
    "Look! The trees move in the wind!",
    "Shall we begin?",
    "Got you, Jake!",
    "This is the greatest of days!",
    "Wow, what a find!",
    "I won!",
    "Got you back!",
    "Let's go again!"
]

In [ ]:
for tok in combine_list:
  n = 0
  for i in range(len(word_level_data)):
    goldStandard = goldStandard_list[i]
    if tok in goldStandard:
      try:
        print(goldStandard, int(id_list[i]), int(sentence_id_list[i]))
      except:
        print(goldStandard, int(id_list[i]))
      n += 1
  print('\n')

  if n == 0:
    print(tok, 'Did not find this sentence')